# 🛍️ Fine-Tuning AI Agents — From Production to Perfection
## Microsoft Build 2026 | LAB231

---

### What you'll see in this notebook:

| # | Section | What we do |
|---|---------|------------|
| 1 | **The Retail Scenario** | Explore a production AI agent handling post-purchase customer service |
| 2 | **Invoke the Prod Agent** | Run GPT-5.4 live and see multi-tool orchestration in action |
| 3 | **Traces → Training Data** | See how production traces become distillation & evaluation data |
| 4 | **Base Model Evaluation** | Run evals across 5 models — establish the baseline |
| 5 | **The Leaderboard** | See where base models struggle (the "execution gap") |

---

**The big question:** *Can we get GPT-5.4 quality from smaller, cheaper models?*

By the end of this lab, we'll have improved models using:
- 📚 **SFT Distillation** — teach students from the teacher's traces
- 🎯 **Reinforcement Fine-Tuning** — let models learn from trial-and-error
- 🔧 **Low-Level finetuning APIs** — maximum control on open-source models

> 💡 **Speaker notes & detailed script:** [`docs/1_introduction.md`](../docs/1_introduction.md)

---

## 🤖 The Retail Agent — Post-Purchase Resolution

Our production agent handles **post-purchase customer service** — returns, exchanges, replacements, and denials.  
It's powered by **GPT-5.4** and orchestrates **6 tools** to resolve each scenario:

```
📦 get_order_details → 🚚 get_fulfillment_status → 📋 check_resolution_policy
    → 📊 check_inventory (exchanges) → 💰 calculate_resolution → ✅ submit_resolution
```

### Why is this hard?

| Challenge | Example |
|-----------|--------|
| Multi-step orchestration | 3–6 tool calls per scenario, strict ordering |
| Complex policy rules | Loyalty tiers × categories × delivery status |
| Precise calculations | Restocking fees, credits, partial refunds |
| Edge cases | Sale items, lost packages, defective items |

### 🔗 Explore live:

| Resource | Link |
|----------|------|
| 🌐 **Tool Demo Console** | [retail-tools-omkarm.azurewebsites.net/demo](https://retail-tools-omkarm.azurewebsites.net/demo) |
| 📄 **Function App Source** | [`tools/retail-tools/function_app.py`](../tools/retail-tools/function_app.py) |
| 🤖 **Agent Source Code** | [`agents/retail/main.py`](../agents/retail/main.py) |

---

### ▶️ Invoke the Production Agent

Run these commands in the terminal to see the agent in action:

```bash
# show status of the "retail" agent
azd ai agent retail show

# Invoke the production agent (GPT-5.4)
azd ai agent invoke retail "Noah Brown. I changed my mind on the Bluetooth Speaker from ORD-010, can I return it?"
```

**What to observe:**
- The agent calls 4+ tools in sequence automatically
- Final response includes: action, refund amount, restocking fee, explanation
- GPT-5.4 handles this correctly — *but at what cost?*

> 💡 **Also show:** Open the agent in [Azure AI Foundry Playground](https://ai.azure.com) → select `retail-prod` → send the same message

---

## 🔍 From Traces to Training Data

Every agent invocation generates **traces** in Azure AI Foundry — the complete record of what the agent did.

### What traces give us:

| Superpower | How |
|-----------|-----|
| 📚 **Distillation data** | GPT-5.4's correct outputs become training examples for smaller models |
| 📏 **Evaluation data** | Add ground truth → measure any model against the same bar |

### 🎯 In Foundry UI:

1. Open **[Azure AI Foundry](https://ai.azure.com)** → your project
2. Navigate to **Tracing** tab
3. Find the trace from the invocation above
4. Observe the full tool-call chain:
   - `get_order_details` → `check_resolution_policy` → `calculate_resolution` → `submit_resolution`
   - Latency per step, token usage, inputs/outputs
5. **Export traces** → Create evaluation dataset

> 🔑 *"These traces are gold — they tell us what the right answer looks like.  
> We use them both to **teach** smaller models (SFT) and to **measure** all models (evals)."*

---

## ⚙️ Setup & Configuration

Now we connect to the project and set up for running evaluations programmatically.

In [41]:
import os
import json
import requests
import time
from datetime import datetime
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from dotenv import load_dotenv

load_dotenv()

PROJECT_ENDPOINT = os.getenv("PROJECT_ENDPOINT")
if not PROJECT_ENDPOINT:
    raise ValueError("PROJECT_ENDPOINT not set. Copy .env.example → .env and fill in your endpoint.")

# Parse account/project from endpoint
parts = PROJECT_ENDPOINT.split("/")
ACCOUNT = parts[2].split(".")[0]
PROJECT = parts[-1]
BASE_URL = f"{PROJECT_ENDPOINT}/openai"

credential = DefaultAzureCredential()

def get_headers():
    token = credential.get_token("https://ai.azure.com/.default").token
    return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

print(f"✅ Connected to: {ACCOUNT}/{PROJECT}")

✅ Connected to: ai-account-44mf5lkxqssxm/ai-project-omi-build26-azd-env


---

## 📊 Base Model Evaluation

Let's establish the baseline — how well do **off-the-shelf models** handle our retail task?

**Evaluation setup:**
- **Dataset:** 62 validation scenarios ([`data/retail_eval.jsonl`](../data/retail_eval.jsonl)) — never seen during training
- **Grader:** Custom Python grader ([`scripts/retail_grader_response.py`](../scripts/retail_grader_response.py))
  - Action correctness: **50%** (did it pick the right resolution?)
  - Financial accuracy: **30%** (are the amounts correct?)
  - Format compliance: **20%** (structured output?)
- **Built-in evaluators:** IntentResolution + TaskCompletion (GPT-5.4 as judge)

**Models under test:**

| Model | Role | Cost tier |
|-------|------|----------|
| GPT-5.4 | Production (teacher) | $$$$$ |
| o4-mini | Reasoning model | $$$ |
| GPT-4.1 | Standard | $$ |
| GPT-4.1-mini | Compact | $ |
| GPT-4.1-nano | Smallest | ¢ |

In [42]:
# Initialize AI Project Client & upload validation dataset
client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")
DATASET_NAME = f"retail-eval-base-{TIMESTAMP}"
EVAL_NAME = f"retail-base-eval-{TIMESTAMP}"
# EVAL_NAME = f"retail-base_sft-eval-{TIMESTAMP}"
# EVAL_NAME = f"retail-base-sft-rft-eval-{TIMESTAMP}"


dataset = client.datasets.upload_file(
    file_path="../data/retail_eval.jsonl",
    name=DATASET_NAME,
    version="1.0"
)
DATASET_URI = f"azureai://accounts/{ACCOUNT}/projects/{PROJECT}/data/{dataset.name}/versions/{dataset.version}"
print(f"✅ Dataset uploaded: {dataset.name}")
print(f"   URI: {DATASET_URI}")

✅ Dataset uploaded: retail-eval-base-20260531-052425
   URI: azureai://accounts/ai-account-44mf5lkxqssxm/projects/ai-project-omi-build26-azd-env/data/retail-eval-base-20260531-052425/versions/1.0


In [43]:
# Load the custom grader
with open("../scripts/retail_grader_response.py", "r") as f:
    GRADER_SOURCE = f.read()

print(f"✅ Loaded grader ({len(GRADER_SOURCE)} chars)")
print(f"   Scoring: action (50%) + financial (30%) + format (20%)")
print(f"   Pass threshold: 0.8")

✅ Loaded grader (20778 chars)
   Scoring: action (50%) + financial (30%) + format (20%)
   Pass threshold: 0.8


In [44]:
# Create evaluation definition with 3 criteria
eval_body = {
    "name": EVAL_NAME,
    "data_source_config": {
        "type": "custom",
        "include_sample_schema": True,
        "item_schema": {
            "type": "object",
            "properties": {
                "messages": {"type": "array"},
                "expected_resolution": {"type": "string"},
                "expected_actions": {"type": "object"},
                "expected_amounts": {"type": "object"}
            }
        }
    },
    "testing_criteria": [
        {
            "name": "retail_quality",
            "type": "python",
            "source": GRADER_SOURCE,
            "pass_threshold": 0.8
        },
        {
            "type": "azure_ai_evaluator",
            "name": "IntentResolution",
            "evaluator_name": "builtin.intent_resolution",
            "initialization_parameters": {"deployment_name": "gpt-5.4"},
            "data_mapping": {
                "query": "{{item.messages}}",
                "response": "{{sample.output_text}}",
                "ground_truth": "{{item.expected_resolution}}"
            }
        },
        {
            "type": "azure_ai_evaluator",
            "name": "TaskCompletion",
            "evaluator_name": "builtin.task_completion",
            "initialization_parameters": {"deployment_name": "gpt-5.4"},
            "data_mapping": {
                "query": "{{item.messages}}",
                "response": "{{sample.output_text}}",
                "ground_truth": "{{item.expected_resolution}}"
            }
        }
    ]
}

url = f"{BASE_URL}/evals?api-version=2025-11-15-preview"
resp = requests.post(url, headers=get_headers(), json=eval_body)
if resp.status_code in [200, 201]:
    eval_id = resp.json()["id"]
    print(f"✅ Evaluation created: {eval_id}")
    print(f"   Criteria: retail_quality (Python) + IntentResolution + TaskCompletion")
else:
    print(f"❌ Failed: {resp.status_code}")
    print(resp.text[:300])


✅ Evaluation created: eval_33c87fdec63e45cfb3eeed4ea3a66030
   Criteria: retail_quality (Python) + IntentResolution + TaskCompletion


In [45]:
# Base models only — no fine-tuned variants
BASE_MODELS = {
    "gpt-5.4":      ("retail-gpt-5-4", None),
    "o4-mini":      ("retail-o4-mini", None),
    "gpt-4.1":      ("retail-gpt-4-1", None),
    "gpt-4.1-mini": ("retail-gpt-4-1-mini", None),
    "gpt-4.1-nano": ("retail-gpt-4-1-nano", None),
    # "gpt-4.1-mini-finetuned": ("retail-gpt-4-1-mini-finetuned", None),
    # "gpt-4.1-nano-finetuned": ("retail-gpt-4-1-nano-finetuned", None),
    # "o4-mini-finetuned":      ("retail-o4-mini-finetuned", None)
}

run_ids = {}
for display_name, (agent_name, version) in BASE_MODELS.items():
    target = {"type": "azure_ai_agent", "name": agent_name}
    if version:
        target["version"] = version

    run_body = {
        "name": f"eval-{display_name}-{TIMESTAMP}",
        "data_source": {
            "type": "azure_ai_target_completions",
            "source": {"type": "file_id", "id": DATASET_URI},
            "target": target,
            "input_messages": {"type": "item_reference", "item_reference": "item.messages"}
        }
    }
    url = f"{BASE_URL}/evals/{eval_id}/runs?api-version=2025-11-15-preview"
    resp = requests.post(url, headers=get_headers(), json=run_body)
    if resp.status_code in [200, 201]:
        run_ids[display_name] = resp.json()["id"]
        print(f"  🚀 {display_name:12s} → {run_ids[display_name][:8]}...")
    else:
        print(f"  ❌ {display_name}: {resp.status_code} - {resp.text[:150]}")

print(f"\n✅ {len(run_ids)}/{len(BASE_MODELS)} eval runs launched")


  🚀 gpt-5.4      → evalrun_...
  🚀 o4-mini      → evalrun_...
  🚀 gpt-4.1      → evalrun_...
  🚀 gpt-4.1-mini → evalrun_...
  🚀 gpt-4.1-nano → evalrun_...

✅ 5/5 eval runs launched


---

## 🏆 The Leaderboard — Base Model Results

Let's pull results from a completed evaluation run and see how the models stack up.

> 🖥️ **After this cell:** Switch to [Azure AI Foundry → Evaluations](https://ai.azure.com) to show the visual leaderboard and per-scenario breakdown.

In [22]:
# Fetch results from a completed eval run
COMPLETED_EVAL_NAME = "retail-base-eval-20260530-185137"

# List evals to find the completed one
url = f"{BASE_URL}/evals?api-version=2025-11-15-preview"
resp = requests.get(url, headers=get_headers())
evals = resp.json().get("data", [])
target_eval = next((e for e in evals if e["name"] == COMPLETED_EVAL_NAME), None)

if not target_eval:
    # Fall back to the eval we just created
    target_eval_id = eval_id
    print(f"Using current eval: {eval_id}")
else:
    target_eval_id = target_eval["id"]
    print(f"✅ Found eval: {COMPLETED_EVAL_NAME}")
    print(f"   ID: {target_eval_id}")


✅ Found eval: retail-base-eval-20260530-185137
   ID: eval_a809a938459f4700b668a830b792a813


In [23]:
# Fetch all runs and build a detailed leaderboard
url = f"{BASE_URL}/evals/{target_eval_id}/runs?api-version=2025-11-15-preview"
resp = requests.get(url, headers=get_headers())
runs = resp.json().get("data", [])

# Parse results for each run
results = []
for run in runs:
    name = run.get("name", "unknown")
    status = run.get("status", "unknown")
    result_counts = run.get("result_counts", {})
    
    # Parse all criteria scores
    scores = {}
    for criteria in run.get("per_testing_criteria_results", []):
        cname = criteria.get("testing_criteria", "")
        passed = criteria.get("passed", 0)
        failed = criteria.get("failed", 0)
        total = passed + failed
        scores[cname] = passed / total if total > 0 else 0
    
    results.append({
        "name": name,
        "status": status,
        "retail_quality": scores.get("retail_quality", 0),
        "intent": scores.get("IntentResolution", 0),
        "task": scores.get("TaskCompletion", 0),
        "total": result_counts.get("total", 0),
        "report_url": run.get("report_url", "")
    })

# Sort by retail_quality descending
results.sort(key=lambda x: x["retail_quality"], reverse=True)

# Print leaderboard
print("\n" + "=" * 80)
print("  MULTI-MODEL LEADERBOARD — Base vs Fine-Tuned")
print("=" * 80)
print(f"\n{'#':<4} {'Model':<22} {'retail_quality':<16} {'Intent':<10} {'Task':<10} {'Status'}")
print("-" * 75)

for i, r in enumerate(results, 1):
    rq = r['retail_quality']
    bar = chr(9608) * int(rq * 15)
    print(f"{i:<4} {r['name']:<22} {rq:>5.1%} {bar:<15} {r['intent']:>5.1%}    {r['task']:>5.1%}    {r['status']}")

print(f"\n{'─' * 75}")
print(f"Criteria: retail_quality = custom Python grader | Intent = IntentResolution | Task = TaskCompletion")
print(f"\n🔗 View detailed results in Azure AI Foundry:")
if results and results[0].get("report_url"):
    # Extract base URL (before /run/)
    base_url = results[0]["report_url"].split("/run/")[0]
    print(f"   {base_url}")



  MULTI-MODEL LEADERBOARD — Base vs Fine-Tuned

#    Model                  retail_quality   Intent     Task       Status
---------------------------------------------------------------------------
1    eval-o4-mini-20260530-185137 72.6% ██████████      79.0%    32.3%    completed
2    eval-gpt-4.1-20260530-185137 58.1% ████████        85.5%    48.4%    completed
3    eval-gpt-5.4-20260530-185137 58.1% ████████        80.6%    25.8%    completed
4    eval-gpt-4.1-mini-20260530-185137 41.9% ██████          72.6%    27.4%    completed
5    eval-gpt-4.1-nano-20260530-185137  1.6%                 77.4%    40.3%    completed

───────────────────────────────────────────────────────────────────────────
Criteria: retail_quality = custom Python grader | Intent = IntentResolution | Task = TaskCompletion

🔗 View detailed results in Azure AI Foundry:
   https://ai.azure.com/nextgen/r/hWyA_fFOQ2u0NPvESpED9w,rg-omi-build26-azd-env,,ai-account-44mf5lkxqssxm,ai-project-omi-build26-azd-env/build/evalu

---

## 🎯 Key Takeaway: The Execution Gap

- Base models (**gpt-4-1, gpt-5-4, o4-mini, qwen3-32b**) understand intent but still fail at execution.
- Fine-tuned models (**qwen3-32b-finetuned-rl, o4-mini-finetuned-rl, gpt-4-1-mini-finetuned-sft**) show dramatic improvement.
- The best result, **qwen3-32b-finetuned-rl**, improves **retail_quality** from **58.1% → 86.9%**.

### The Promise: 3 Ways to Fix This

| Approach | Difficulty | What it does | Notebook |
|----------|-----------|--------------|----------|
| 📚 **SFT Distillation** | Easiest API | Show smaller models GPT-5.4's correct answers | [`2_sft_distillation.ipynb`](./2_sft_distillation.ipynb) |
| 🎯 **RFT (Foundry SDK)** | Powerful SDK | Let o4-mini learn from trial-and-error with a grader | [`3_rft_foundry.ipynb`](./3_rft_foundry.ipynb) |
| 🔧 **RFT (Low-Level)** | Maximum control | Fine-grained training on OSS Qwen3-32B | [`4_rft_lowlevel.ipynb`](./4_rft_lowlevel.ipynb) |

---

**Next → [2_sft_distillation.ipynb](./2_sft_distillation.ipynb)** — Distill GPT-5.4 into smaller, cheaper models that match or beat the teacher.
